In [1]:
%%capture
!pip install facenet-pytorch

In [180]:
from facenet_pytorch import MTCNN, InceptionResnetV1, fixed_image_standardization, training
import torch
from torch.utils.data import DataLoader, SubsetRandomSampler
from torchvision import datasets, transforms
from torchvision.transforms import v2
from torchsummary import summary
from torch import optim
from torch.optim.lr_scheduler import MultiStepLR
from torch.utils.tensorboard import SummaryWriter

import numpy as np
import pandas as pd
import os
%matplotlib inline
import matplotlib.pyplot as plt
import glob
import re

import PIL
from PIL import ImageFile, Image
ImageFile.LOAD_TRUNCATED_IMAGES = True

workers = 0 if os.name == 'nt' else 4

## Load models

In [4]:
# set device
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('Current device: {}'.format(device))

Current device: cuda:0


In [6]:
# Load model
'''
This model is used to detect faces and it returns the face (cropped)

image_size: output image size
margin: margin of the bounding box added in the ouput image
min_face_size: minimum face size to search within the image
'''
mtcnn = MTCNN(
    image_size=160,
    margin=0,
    min_face_size=20,
    thresholds=[0.6, 0.7, 0.7],
    factor=0.709,
    post_process=True,
    device=device
)

In [7]:
# Load model
'''
The cropped faces are passed as input in the CNN and we get an embedding for each face.
Important to set the model at .eval()
'''
resnet = InceptionResnetV1(pretrained='vggface2').eval().to(device)

  0%|          | 0.00/107M [00:00<?, ?B/s]

In [12]:
# # model architecture and summary
# summary(resnet, (3, 160, 160))

## Load Data

In [ ]:
!rmdir /content/data/test_images/.ipynb_checkpoints

In [42]:
def collate_fn(x):
    return x[0]

In [64]:
dataset = datasets.ImageFolder('/content/data/test_images')
dataset.idx_to_class = {i:c for c, i in dataset.class_to_idx.items()}
loader = DataLoader(dataset, collate_fn=collate_fn, num_workers=workers)

## Face detection using MTCNN

In [76]:
aligned = []
names = []
for x, y in loader:
    x_aligned, prob = mtcnn(x, return_prob=True)
    if x_aligned is not None:
        print('Face detected with probability: {:8f}'.format(prob))
        aligned.append(x_aligned)
        names.append(dataset.idx_to_class[y])


/usr/local/lib/python3.10/dist-packages/torch/utils/data/dataloader.py:558: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(
/usr/lib/python3.10/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()


Face detected with probability: 0.999983
Face detected with probability: 0.999954
Face detected with probability: 0.999733


/usr/lib/python3.10/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()


Face detected with probability: 0.999876


## Calculate image embedding

In [77]:
aligned = torch.stack(aligned).to(device)
embeddings = resnet(aligned).detach().cpu()

In [78]:
# embeddings distance
dists = [[(e1 - e2).norm().item() for e2 in embeddings] for e1 in embeddings]
print(pd.DataFrame(dists, columns=names, index=names))

                angelina_jolie  bradley_cooper  kate_siegel  paul_rudd
angelina_jolie        0.000000        1.427359     0.887728   1.434377
bradley_cooper        1.427359        0.000000     1.279511   1.034571
kate_siegel           0.887728        1.279511     0.000000   1.388992
paul_rudd             1.434377        1.034571     1.388992   0.000000


## Finetuning on a new image

In [164]:
data_dir = '/content/data/train_images'
workers = 0 if os.name == 'nt' else 8
random_people = len(glob.glob(os.path.join('/content/data/train_images/random_people/*.jpg')))

print('There are {} random people.'.format(random_people))

There are 4 random people.


#### Extract the face from the training dataset

In [165]:
!rmdir /content/data/train_images/.ipynb_checkpoints

rmdir: failed to remove '/content/data/train_images/.ipynb_checkpoints': No such file or directory


In [166]:
dataset = datasets.ImageFolder(data_dir, transform=transforms.Resize((512, 512)))
dataset.samples = [
    (p, p.replace(data_dir, data_dir + '_cropped'))
        for p, _ in dataset.samples
]

loader = DataLoader(
    dataset,
    num_workers=workers,
    batch_size=1,
    collate_fn=training.collate_pil
)

/usr/local/lib/python3.10/dist-packages/torch/utils/data/dataloader.py:558: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(


In [167]:
for i, (x, y) in enumerate(loader):
    mtcnn(x, save_path=y)
    print('\rBatch {} of {}'.format(i + 1, len(loader)), end='')

/usr/lib/python3.10/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()


Batch 3 of 5

/usr/lib/python3.10/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()


Batch 5 of 5

#### Data Augmentation

In [168]:
aug_transform = transforms.Compose([
    v2.ToTensor(),
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomHorizontalFlip(p=0.5),
    v2.ColorJitter(brightness=.5, hue=.3),
    v2.GaussianBlur(kernel_size=(5, 9), sigma=(0.1, 5.)), v2.RandomAutocontrast(),
    v2.ToPILImage()])

/usr/local/lib/python3.10/dist-packages/torchvision/transforms/v2/_deprecated.py:41: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.
  warnings.warn(


In [186]:
new_path = y[0].replace('train_images_cropped', 'train_images_augmented')
new_path

'/content/data/train_images_augmented/random_people/4.jpg'

In [187]:
new_path = re.sub(r'(\d+)\.jpg', r'{}.jpg'.format(5), new_path)
new_path

'/content/data/train_images_augmented/random_people/5.jpg'

In [188]:
num_augs = 64
random_people_counter = 0
for _, y in loader:
  im = Image.open(y[0])
  try:
    os.mkdir('/'.join(y[0].replace('train_images_cropped', 'train_images_augmented').split('/')[:-1]))
  except:
    print('Directory already exists')
  if 'random_people' in y[0]:
    for _ in range(num_augs//random_people):
      new_im = aug_transform(im)
      new_path = y[0].replace('train_images_cropped', 'train_images_augmented')
      new_path = re.sub(r'(\d+)\.jpg', r'{}.jpg'.format(random_people_counter), new_path)
      # new_im.save(y[0].replace('train_images_cropped', 'train_images_augmented').replace('*.jpg', '{}.jpg'.format(random_people_counter)))
      new_im.save(new_path)
      random_people_counter += 1
  else:
    for i in range(num_augs):
      new_im = aug_transform(im)
      new_im.save(y[0].replace('train_images_cropped', 'train_images_augmented').replace('1.jpg', '{}.jpg'.format(i)))


/usr/local/lib/python3.10/dist-packages/torch/utils/data/dataloader.py:558: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(
/usr/lib/python3.10/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()


Directory already exists
Directory already exists
Directory already exists
Directory already exists


/usr/lib/python3.10/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()


Directory already exists


#### Finetune the model

In [190]:
data_dir = '/content/data/train_images_augmented'

batch_size = 16
epochs = 8

resnet = InceptionResnetV1(
    classify=True,
    pretrained='vggface2',
    num_classes=len(dataset.class_to_idx) if len(dataset.class_to_idx)>1 else 2
).to(device)

print('Model is trained on {} classes.'.format(len(dataset.class_to_idx)))

Model is trained on 2 classes.


In [191]:
optimizer = optim.Adam(resnet.parameters(), lr=0.001)
scheduler = MultiStepLR(optimizer, [5, 10])

trans = transforms.Compose([
    np.float32,
    transforms.ToTensor(),
    fixed_image_standardization
])

dataset = datasets.ImageFolder(data_dir, transform=trans)
img_inds = np.arange(len(dataset))
np.random.shuffle(img_inds)
train_inds = img_inds[:int(0.8 * len(img_inds))]
val_inds = img_inds[int(0.8 * len(img_inds)):]

train_loader = DataLoader(
    dataset,
    num_workers=workers,
    batch_size=batch_size,
    sampler=SubsetRandomSampler(train_inds)
)
val_loader = DataLoader(
    dataset,
    num_workers=workers,
    batch_size=batch_size,
    sampler=SubsetRandomSampler(val_inds)
)

/usr/local/lib/python3.10/dist-packages/torch/utils/data/dataloader.py:558: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(


In [194]:
loss_fn = torch.nn.CrossEntropyLoss()
metrics = {
    'fps': training.BatchTimer(),
    'acc': training.accuracy
}

In [195]:
writer = SummaryWriter()
writer.iteration, writer.interval = 0, 10

print('\n\nInitial')
print('-' * 10)
resnet.eval()
training.pass_epoch(
    resnet, loss_fn, val_loader,
    batch_metrics=metrics, show_running=True, device=device,
    writer=writer
)

for epoch in range(epochs):
    print('\nEpoch {}/{}'.format(epoch + 1, epochs))
    print('-' * 10)

    resnet.train()
    training.pass_epoch(
        resnet, loss_fn, train_loader, optimizer, scheduler,
        batch_metrics=metrics, show_running=True, device=device,
        writer=writer
    )

    resnet.eval()
    training.pass_epoch(
        resnet, loss_fn, val_loader,
        batch_metrics=metrics, show_running=True, device=device,
        writer=writer
    )

writer.close()



Initial
----------


/usr/local/lib/python3.10/dist-packages/torch/utils/data/dataloader.py:558: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(
/usr/lib/python3.10/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()
/usr/lib/python3.10/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()


Valid |     2/2    | loss:    0.7435 | fps:   22.7196 | acc:    0.6125   

Epoch 1/8
----------
Train |     7/7    | loss:    0.2946 | fps:  100.9575 | acc:    0.8929   
Valid |     2/2    | loss:  187.3273 | fps:  109.5840 | acc:    0.4375   

Epoch 2/8
----------
Train |     7/7    | loss:    0.0632 | fps:  138.8281 | acc:    0.9524   
Valid |     2/2    | loss:    8.6171 | fps:  142.0654 | acc:    0.6000   

Epoch 3/8
----------
Train |     7/7    | loss:    0.0206 | fps:  142.9834 | acc:    0.9911   
Valid |     2/2    | loss:    0.0082 | fps:  149.4449 | acc:    1.0000   

Epoch 4/8
----------
Train |     7/7    | loss:    0.0740 | fps:  139.9915 | acc:    0.9821   
Valid |     2/2    | loss:    0.0404 | fps:  150.1697 | acc:    0.9500   

Epoch 5/8
----------
Train |     7/7    | loss:    0.0024 | fps:  138.8817 | acc:    1.0000   
Valid |     2/2    | loss:    0.0239 | fps:  144.8489 | acc:    1.0000   

Epoch 6/8
----------
Train |     7/7    | loss:    0.0368 | fps:  137.4312 

## Test the model

In [202]:
!rmdir /content/data/val_images/.ipynb_checkpoints

In [203]:
data_dir = '/content/data/val_images'
dataset = datasets.ImageFolder(data_dir)
dataset.idx_to_class = {i:c for c, i in dataset.class_to_idx.items()}
loader = DataLoader(dataset, collate_fn=collate_fn, num_workers=workers)

In [204]:
aligned = []
names = []
for x, y in loader:
    x_aligned, prob = mtcnn(x, return_prob=True)
    if x_aligned is not None:
        print('Face detected with probability: {:8f}'.format(prob))
        aligned.append(x_aligned)
        names.append(dataset.idx_to_class[y])


Face detected with probability: 0.985002
Face detected with probability: 0.999986


In [208]:
resnet.eval()
print('Done')

Done


In [207]:
aligned = torch.stack(aligned).to(device)
embeddings = resnet(aligned).detach().cpu()

# embeddings distance
dists = [[(e1 - e2).norm().item() for e2 in embeddings] for e1 in embeddings]
print(pd.DataFrame(dists, columns=names, index=names))

                   goku  random_people
goku           0.000000       9.998696
random_people  9.998696       0.000000
